<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day09-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 9 lab: training networks yourself on MNIST {.unnumbered}

The Day 8 lab gave you a ready-made `train_model` function. In this lab
you **write the training loop yourself**, and then use it to run the
experiments behind the Day 9 page:

- which **loss function** works for a classifier;
- what happens if every weight starts at the **same value**;
- how **depth** and the choice of **activation** affect the gradient that
  reaches the first layer (vanishing gradients);
- **SGD vs. Adam**, and a wider hidden layer;
- a small **hyperparameter search**, chosen on the validation set, with
  the test set used exactly once.

Every code cell with a `...` (three dots) needs a piece of real code,
described by its `# TODO` comment. Work through the cells **in order,
top to bottom**. An unfixed cell will either print `Ellipsis` or raise
an error pointing at the line that still needs fixing. That is expected.

**Runtime:** a few minutes on Colab's free CPU. Keep `SEED = 0` so that
your numbers match the lab quiz. Neural-network training still varies a
little between machines, so the quiz accepts a range, and you upload
your notebook at the end.

In [ ]:
import os, tempfile, itertools
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, TensorDataset

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(max(1, os.cpu_count() or 1))

# MNIST is downloaded to a temporary directory, not next to this notebook
DATA_DIR = os.path.join(tempfile.gettempdir(), "kb8029_mnist")

## 1. The data (as in the Day 8 lab)

500 training images and 100 validation images per digit, drawn from the
official 60,000-image training set, and the official 10,000-image test
set, untouched until Section 7. Pixels are standardized with statistics
from the **training images only** (Day 8). Each image is flattened to a
vector of 784 numbers.

In [ ]:
mnist_train = torchvision.datasets.MNIST(DATA_DIR, train=True, download=True)
mnist_test = torchvision.datasets.MNIST(DATA_DIR, train=False, download=True)
X_all = mnist_train.data.numpy().astype(np.float32) / 255.0
y_all = mnist_train.targets.numpy()
X_test_raw = mnist_test.data.numpy().astype(np.float32) / 255.0
y_test_np = mnist_test.targets.numpy()

rng = np.random.RandomState(SEED)
train_idx, val_idx = [], []
for digit in range(10):
    idx = rng.permutation(np.where(y_all == digit)[0])
    val_idx.append(idx[:100])
    train_idx.append(idx[100:600])
train_idx, val_idx = np.concatenate(train_idx), np.concatenate(val_idx)

mu, sd = X_all[train_idx].mean(), X_all[train_idx].std()     # training images only
to_tensor = lambda X: torch.tensor(((X - mu) / sd).reshape(len(X), -1))
X_train, X_val, X_test = to_tensor(X_all[train_idx]), to_tensor(X_all[val_idx]), to_tensor(X_test_raw)
y_train, y_val, y_test = torch.tensor(y_all[train_idx]), torch.tensor(y_all[val_idx]), torch.tensor(y_test_np)
print("train:", tuple(X_train.shape), " val:", tuple(X_val.shape), " test:", tuple(X_test.shape))

## 2. A network builder, and the training loop

`make_net` builds a fully connected network: `depth` hidden layers of
`width` units, each followed by an activation, and a final `Linear` layer
with 10 output logits. Fill in the one line that adds a hidden layer.

In [ ]:
def make_net(depth=1, width=10, activation=nn.Tanh):
    layers, n_in = [], 784
    for _ in range(depth):
        layers += ...  # TODO: a list with nn.Linear(n_in, width) followed by activation()
        n_in = width
    layers.append(nn.Linear(n_in, 10))
    return nn.Sequential(*layers)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

print(make_net())

Now the heart of the lab: the **training loop**. A `DataLoader` shuffles
the training set every epoch and serves it in mini-batches. For every
batch, the same four steps from the Day 9 page:

1. clear the old gradients (`optimizer.zero_grad()`);
2. forward pass and loss (`loss = loss_fn(model(xb), yb)`);
3. backpropagation (`loss.backward()`);
4. update every weight (`optimizer.step()`).

After each epoch, record training and validation accuracy.

Some loss functions (MSE, MAE) compare the network's *probabilities* with
a **one-hot** target (a vector of ten numbers, 1 for the true digit and 0
elsewhere) instead of taking the class index directly. The argument
`one_hot=True` handles that: the loop then applies softmax to the logits
and compares with the one-hot vector.

In [ ]:
def train(model, loss_fn, optimizer, epochs=20, batch_size=64, one_hot=False, seed=SEED):
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            # TODO: the four steps. Hint: if one_hot is True, the loss is
            #   loss_fn(torch.softmax(model(xb), dim=1), nn.functional.one_hot(yb, 10).float())
            # otherwise it is loss_fn(model(xb), yb).
            ...
        history["train_acc"].append(accuracy(model, X_train, y_train))
        history["val_acc"].append(accuracy(model, X_val, y_val))
    return history

torch.manual_seed(SEED)
model = make_net()
hist = train(model, nn.CrossEntropyLoss(), torch.optim.SGD(model.parameters(), lr=0.1), epochs=20)
print(f"after 20 epochs: train accuracy {hist['train_acc'][-1]:.3f}, validation accuracy {hist['val_acc'][-1]:.3f}")

## 3. Which loss function?

Train the same network (10 tanh hidden units, SGD, learning rate 0.1, 20
epochs) three times, with three losses:

- **cross-entropy**, the standard choice for classification (Day 9 page);
- **mean squared error** between the softmax probabilities and the one-hot target;
- **mean absolute error** between the same two.

Re-create the model before each run, so that every run starts from
fresh weights.

In [ ]:
losses = {
    "cross-entropy": (nn.CrossEntropyLoss(), False),
    "MSE": (nn.MSELoss(), True),
    "MAE": (nn.L1Loss(), True),
}
loss_results = {}
for name, (loss_fn, one_hot) in losses.items():
    torch.manual_seed(SEED)
    m = make_net()
    h = train(m, loss_fn, torch.optim.SGD(m.parameters(), lr=0.1), epochs=20, one_hot=one_hot)
    loss_results[name] = h
    print(f"{name:14s} validation accuracy after 20 epochs: {h['val_acc'][-1]:.3f}")
    if name == "cross-entropy":
        ce_model = m

plt.figure(figsize=(6, 3.5))
for name, h in loss_results.items():
    plt.plot(h["val_acc"], label=name)
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.show()

Now look at the **values** of the two regression losses for the
cross-entropy model's own predictions on the validation set: the softmax
probabilities against the one-hot targets, averaged over all images and
all ten outputs.

In [ ]:
with torch.no_grad():
    probs = torch.softmax(ce_model(X_val), dim=1)
targets = nn.functional.one_hot(y_val, 10).float()
mse_value = ...  # TODO: mean of the squared differences
mae_value = ...  # TODO: mean of the absolute differences
print(f"MSE = {mse_value:.4f}   MAE = {mae_value:.4f}")

## 4. What if every weight starts at 1?

Take the same network, but before training set **every** weight and bias
to 1. Then train as before and look at the hidden layer's weights: the
10 rows of `model[0].weight`, one row of 784 weights per hidden unit.

In [ ]:
torch.manual_seed(SEED)
ones_model = make_net()
with torch.no_grad():
    for p in ones_model.parameters():
        ...  # TODO: set every entry of p to 1.0 (hint: p.fill_)
h_ones = train(ones_model, nn.CrossEntropyLoss(), torch.optim.SGD(ones_model.parameters(), lr=0.1), epochs=20)
W = ones_model[0].weight.detach()
max_row_difference = ...  # TODO: largest absolute difference between any hidden unit's weights and unit 0's
print(f"validation accuracy: {h_ones['val_acc'][-1]:.3f}")
print(f"largest difference between hidden units' weights after training: {max_row_difference:.2e}")

## 5. Depth and vanishing gradients

Build networks with 1, 2, 4 and 8 hidden layers of 10 units, with tanh,
sigmoid and ReLU activations. For each, first measure the mean absolute
**gradient of the loss with respect to the first layer's weights** at
initialization, on the whole training set. Then train for 10 epochs (SGD,
learning rate 0.1) and record validation accuracy.

In [ ]:
depth_results = {}
for activation in [nn.Tanh, nn.Sigmoid, nn.ReLU]:
    for depth in [1, 2, 4, 8]:
        torch.manual_seed(SEED)
        m = make_net(depth=depth, activation=activation)
        loss = nn.CrossEntropyLoss()(m(X_train), y_train)
        first_layer_grad = ...  # TODO: gradient of loss w.r.t. m[0].weight (hint: torch.autograd.grad(...)[0])
        grad_size = first_layer_grad.abs().mean().item()
        h = train(m, nn.CrossEntropyLoss(), torch.optim.SGD(m.parameters(), lr=0.1), epochs=10)
        depth_results[(activation.__name__, depth)] = (grad_size, h["val_acc"][-1])
        print(f"{activation.__name__:8s} depth {depth}: first-layer |gradient| {grad_size:.2e}   "
              f"validation accuracy {h['val_acc'][-1]:.3f}")

## 6. SGD vs. Adam, and a wider hidden layer

Go back to one hidden layer, but make it 128 units wide. Train it for 10
epochs once with plain SGD (learning rate 0.1) and once with **Adam**
(learning rate 0.001), the optimizer used on the Day 9 page.

In [ ]:
wide = make_net(depth=1, width=128)
n_params_wide = ...  # TODO: total number of trainable parameters
print("parameters in the 128-unit network:", n_params_wide)

optimizer_results = {}
for name in ["SGD", "Adam"]:
    torch.manual_seed(SEED)
    m = make_net(depth=1, width=128)
    if name == "SGD":
        opt = torch.optim.SGD(m.parameters(), lr=0.1)
    else:
        opt = ...  # TODO: an Adam optimizer with learning rate 0.001
    h = train(m, nn.CrossEntropyLoss(), opt, epochs=10)
    optimizer_results[name] = h
    print(f"{name:4s}: train accuracy {h['train_acc'][-1]:.3f}, validation accuracy {h['val_acc'][-1]:.3f}")

Does the hidden layer pay off here? The Day 9 page compared every
network against a **linear** baseline. `make_net(depth=0)` has no hidden
layer at all: a single `Linear(784, 10)`, i.e. multi-class logistic
regression. Train it the same way (Adam, learning rate 0.001, 10 epochs).

In [ ]:
torch.manual_seed(SEED)
linear = ...  # TODO: a network with no hidden layer
h_lin = train(linear, nn.CrossEntropyLoss(), torch.optim.Adam(linear.parameters(), lr=1e-3), epochs=10)
n_params_linear = sum(p.numel() for p in linear.parameters())
print(f"linear model: {n_params_linear} parameters, validation accuracy {h_lin['val_acc'][-1]:.3f}")
print(f"128-unit network with Adam: validation accuracy {optimizer_results['Adam']['val_acc'][-1]:.3f}")
print(f"training images per weight in the 128-unit network: {len(X_train) / n_params_wide:.3f}")

## 7. A small hyperparameter search, then the test set once

Try every combination of hidden-layer width (32 or 128) and dropout rate
(0 or 0.3) after the hidden layer, with Adam (learning rate 0.001, 10
epochs). **Choose the best combination on the validation set.** Only
then evaluate that one model on the test set, once. Choosing on the test
set would make the test number optimistic (Day 8).

In [ ]:
def make_dropout_net(width, dropout):
    return nn.Sequential(nn.Linear(784, width), nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, 10))

search = {}
for width, dropout in itertools.product([32, 128], [0.0, 0.3]):
    torch.manual_seed(SEED)
    m = make_dropout_net(width, dropout)
    h = train(m, nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), lr=1e-3), epochs=10)
    search[(width, dropout)] = (h["val_acc"][-1], m)
    print(f"width {width:3d}, dropout {dropout}: validation accuracy {h['val_acc'][-1]:.3f}")

best_config = ...  # TODO: the (width, dropout) with the highest VALIDATION accuracy
best_model = search[best_config][1]
test_accuracy = ...  # TODO: accuracy of best_model on the test set (used once, here)
print(f"\nchosen on validation: width {best_config[0]}, dropout {best_config[1]}")
print(f"test accuracy of that model: {test_accuracy:.3f}")

**Last step:** save your notebook with all outputs (File → Download →
.ipynb) and upload it with the lab quiz on Canvas. If one of your numbers
falls outside the quiz's accepted range, the notebook lets us see whether
that is ordinary run-to-run variation in training.